# Install Ollama and and other dependencies

In [1]:
!apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Install langchain libraries
!pip install langchain
!pip install langchain-ollama
!pip install langchain-community
!pip install langchain-core
!pip install pydantic

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (2,163 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current 

# Run ollama in background and donwload models

In [4]:
import subprocess
import time

ollama_process = subprocess.Popen(["ollama", "serve"],
                                  stdout=subprocess.PIPE,
                                  stderr=subprocess.PIPE)

time.sleep(20)
# Download a model
!ollama pull llama3.2:1b

In [5]:
# Initialize the llm model using langchain
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:1b")

# Tools

In [9]:
from langchain_core.tools import tool

@tool
def square(x : int)-> int:
  """ Returns square of a number """
  return x**2

@tool
def add(a:int, b:int)->int:
  """ Adds two number """
  return a+b

@tool
def get_weather(city: str)-> str :
  """ returns given city weather condition """
  return "rainy weather is predicted"

In [11]:
# Test Tool Calling
result = square.invoke({"x":4})
print(result)


result = add.invoke({"a":4, "b":10})
print(result)

16
14


In [12]:
# attach tools to LLM model
from langchain_ollama import ChatOllama
# create model
llm = ChatOllama(model="llama3.2:1b")

# create list of tools
tools = [square, add, get_weather]

# attach tool to model
llm_w_tools = llm.bind_tools(tools)

In [13]:
response = llm_w_tools.invoke("what is 12 plus 5 square ?")
print(response)

content='' additional_kwargs={} response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-06-24T16:42:53.18668849Z', 'done': True, 'done_reason': 'stop', 'total_duration': 69250188122, 'load_duration': 39406873741, 'prompt_eval_count': 241, 'prompt_eval_duration': 29235055000, 'eval_count': 68, 'eval_duration': 604539000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'} id='lc_run--019efa82-4cf7-7593-9390-ffac71d6df49-0' tool_calls=[{'name': 'square', 'args': {'x': '12'}, 'id': 'e743ba4b-f762-4750-a03c-37c2961ebcb5', 'type': 'tool_call'}, {'name': 'add', 'args': {'b': '5', 'a': '12'}, 'id': '7b301c4e-4125-4bc5-9a2f-73ec300da570', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': 'New York'}, 'id': '42c94976-729e-4e03-b89a-075cca40b6de', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 241, 'output_tokens': 68, 'total_tokens': 309}


In [16]:
print(response.tool_calls)

[{'name': 'square', 'args': {'x': '12'}, 'id': 'e743ba4b-f762-4750-a03c-37c2961ebcb5', 'type': 'tool_call'}, {'name': 'add', 'args': {'b': '5', 'a': '12'}, 'id': '7b301c4e-4125-4bc5-9a2f-73ec300da570', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': 'New York'}, 'id': '42c94976-729e-4e03-b89a-075cca40b6de', 'type': 'tool_call'}]


In [17]:
response = llm_w_tools.invoke("what is the weather in bangalore")
print(response)

content='' additional_kwargs={} response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-06-24T16:47:23.026998919Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1066684298, 'load_duration': 688078178, 'prompt_eval_count': 239, 'prompt_eval_duration': 18808000, 'eval_count': 26, 'eval_duration': 354238000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'} id='lc_run--019efa87-7565-7b30-9d39-f7401fa9cac4-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'bangalore'}, 'id': '4cc30fc4-f309-4d1f-863f-1c65edd0a80c', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 239, 'output_tokens': 26, 'total_tokens': 265}


In [18]:
print(response.tool_calls)

[{'name': 'get_weather', 'args': {'city': 'bangalore'}, 'id': '4cc30fc4-f309-4d1f-863f-1c65edd0a80c', 'type': 'tool_call'}]


In [19]:
response = llm_w_tools.invoke("Explain transformer ")
print(response)

content='}\n\n{\n\n  "name": "add",\n  "parameters": {\n    "a": 10,\n    "b": 2\n  }\n}\n\n {\n\n  "name": "get_weather",\n  "parameters": {\n    "city": "Tokyo"\n  }\n}' additional_kwargs={} response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-06-24T16:50:31.627407572Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1718559426, 'load_duration': 940170400, 'prompt_eval_count': 236, 'prompt_eval_duration': 21962000, 'eval_count': 79, 'eval_duration': 751889000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'} id='lc_run--019efa8a-5389-7ad3-ad88-3a50468d6645-0' tool_calls=[{'name': 'square', 'args': {'x': 5}, 'id': 'efa0cd94-c40c-493d-afda-c47d6cf081a2', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 236, 'output_tokens': 79, 'total_tokens': 315}


In [20]:
print(response.tool_calls)

[{'name': 'square', 'args': {'x': 5}, 'id': 'efa0cd94-c40c-493d-afda-c47d6cf081a2', 'type': 'tool_call'}]
